# IMC Prosperity 4 — Round 1 / Round 2 Market Analysis (V3)

This notebook is a **driver** for the canonical analyzer in `analysis.py`.

Key design points:
- `analysis.py` contains the *exhaustive* analysis. The notebook runs it cell-by-cell
  so you can inspect intermediate results interactively.
- The notebook and the script are guaranteed in sync — every cell calls
  `analysis.sectionN(...)`.
- Outputs go to:
  - `plots/round1/` or `plots/round2/` — full plot set
  - `ai_strategy_context/` — curated AI handoff (`strategy_brief.md`,
    `product_params.json`, `metric_summary.csv`, `visual_index.md`, `plots/`,
    `data_samples/`)

Switch round by changing the `ROUND` variable in the setup cell.


In [ ]:
%matplotlib inline
import importlib, sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else os.getcwd())
import analysis; importlib.reload(analysis)

# ---- choose round (round1 / round2) ----
ROUND = "round2"
analysis.configure_round(ROUND)
print("DATA_DIR:", analysis.DATA_DIR)
print("PLOT_DIR:", analysis.PLOT_DIR)


## Section 0 — Data loading & feature computation

In [ ]:
products, product_dfs, trade_dfs = analysis.load_data()
feat_dfs = {p: analysis.compute_features(product_dfs[p]) for p in products}
for p, df in feat_dfs.items():
    print(f"  {p:30s} rows={len(df):>6,}  cols={len(df.columns)}")


## Section 1 — Product classification (within-day CV + drift detection)
Labels each product as **STABLE / DRIFTING / VOLATILE**, the first branch of
the archetype taxonomy.

In [ ]:
verdicts = analysis.section1(products, feat_dfs)
verdicts

## Section 2 — Fair value estimation (4 proxies compared)
`simple_mid` / `microprice` / `wall_mid` / `mm_mid`. Lowest-std proxy → best
anchor for take/clear/make.

In [ ]:
fv_results = analysis.section2(products, feat_dfs)
fv_results

## Section 3 — Spread & edge calibration
Spread distribution per product with suggested `take_edge ≈ p25/4` and
`passive_edge ≈ p50/3`.

In [ ]:
spread_results = analysis.section3(products, feat_dfs)
spread_results

## Section 4 — Mean reversion vs trend (lag-1..10 ACF, z-score zones)

In [ ]:
acf_results = analysis.section4(products, feat_dfs)
acf_results

## Section 5 — Signal predictiveness (multi-horizon)
Candidate signals: `ret1`, `z20`, `micro_delta`, `wall_delta`, `mm_delta`,
`imbalance`, `spread_chg`. Look at |corr| decay and hit rate.

In [ ]:
signal_results = analysis.section5(products, feat_dfs)

## Section 6 — Counterparty / trade pattern analysis
When `buyer`/`seller` IDs are populated (later rounds), this is where the
trader-ID follower archetype lives.

In [ ]:
analysis.section6(products, feat_dfs, trade_dfs)

## Section 7 — Intraday patterns (day overlay, average path, long-lag ACF)

In [ ]:
analysis.section7(products, feat_dfs)

## Section 8 — Spike detection & event study

In [ ]:
spike_results = analysis.section8(products, feat_dfs)
spike_results

## Section 9 — Cross-product correlation / lead-lag

In [ ]:
analysis.section9(products, feat_dfs)

## Section 10 — Position & inventory risk (take/clear/make simulation)

In [ ]:
pos_results = analysis.section10(products, feat_dfs, spread_results)
pos_results

## Section 11 — Parameter sensitivity grid (take_edge × clear_threshold)

In [ ]:
analysis.section11(products, feat_dfs)

## Section 13 — Extended microstructure
Weighted spread, L1 imbalance, OFI (Cont–Kukanov–Stoikov style), depth
concentration, book slope, quote-change intensity, microprice − mid.

In [ ]:
micro_results = analysis.section13(products, feat_dfs)
micro_results

## Section 14 — Conditional returns by state bucket
For each of `imb_l1`, `imbalance`, `ofi_5`, `micro_minus_mid`, `fair_dev_wall`,
`z20`, `spread`: quintile-bucket the feature, compute E[fwd_ret_h] per
bucket, and flag *monotonic* relationships (|Spearman ρ|≥0.9). This is the
primary conditional-edge discovery layer.

In [ ]:
conditional_results = analysis.section14(products, feat_dfs)

## Section 15 — Regime segmentation (vol × spread state)
Each tick labelled `low/med/high × tight/normal/wide`. Table shows regime
occupancy and average absolute return in each regime.

In [ ]:
regime_results = analysis.section15(products, feat_dfs)

## Section 16 — Execution quality & markout
Hypothetical take events at `wall_mid ± take_edge`, then measure signed
markout at h=1/5/20 and adverse-selection rate. Tells you whether the paper
edge actually survives execution.

In [ ]:
exec_results = analysis.section16(products, feat_dfs, spread_results)
exec_results

## Section 17 — Cross-day robustness
Per-day correlation of each signal with `fwd_ret_1`. Watch for sign flips
and magnitude collapse across days.

In [ ]:
robustness_results = analysis.section17(products, feat_dfs, signal_results)

## Section 18 — AR(1) half-life by day (supplementary to ACF)

In [ ]:
half_life_results = analysis.section18(products, feat_dfs)
half_life_results

## Section 19 — Feature relevance (standardised OLS, rank-only)
Tiny ridge-stabilised OLS of `fwd_ret_1` on standardised features. The R² is
the diagnostic; coefficients are used to **rank** features, not to predict.

In [ ]:
feature_relevance = analysis.section19(products, feat_dfs)
feature_relevance

## Section 12 — Strategy decision dashboard (9-panel summary per product)

In [ ]:
analysis.section12(products, feat_dfs, verdicts, fv_results, spread_results,
                   acf_results, signal_results, spike_results, pos_results)


## Section 20 — Build curated AI strategy context export

Writes to `ai_strategy_context/`:
- `strategy_brief.md` — human/AI summary, per product
- `product_params.json` — compact machine-readable payload
- `metric_summary.csv` — one-row-per-product decision table
- `visual_index.md` — which curated plot answers which question
- `plots/` — curated PNG subset (~20 images)
- `data_samples/` — 200-row CSV samples per product

The full exhaustive notebook output stays in `plots/<round>/`.

In [ ]:
analysis.build_ai_export(products, feat_dfs, verdicts, fv_results, spread_results,
                         acf_results, signal_results, spike_results, pos_results,
                         micro_results, conditional_results, regime_results,
                         exec_results, robustness_results, half_life_results,
                         feature_relevance, trade_dfs)


---
### Next step prompts

- **Alpha discovery** — hand `ai_strategy_context/strategy_brief.md` +
  `product_params.json` + `plots/14A_conditional_returns_*.png` +
  `plots/16A_markout_*.png` to an AI tool with the question
  *"Which conditional relationship in the brief has the strongest tradable edge
  that survives execution markout? Propose a concrete take→clear→make rule."*

- **Strategy implementation** — hand `product_params.json` +
  `strategy_brief.md` Top-3 ideas section + `RUST_BACKTESTER_CONTEXT_FOR_AI_TOOLS.md`
  to Codex with the question *"Implement Round 2 trader in Rust using the
  configured fair proxy, thresholds, and inventory posture from
  product_params.json. Preserve take/clear/make separation."*